# Data collection

In [ ]:
import requests
import time
import pandas as pd
from dotenv import load_dotenv

# Load environment variables from the .env file
load_dotenv()

# Grab the key securely
API_KEY = os.getenv("RIOT_API_KEY")

if not API_KEY:
    print("WARNING: API Key not found. Check your .env file!")

headers = {
    "X-Riot-Token": API_KEY
}

# Riot API uses two different routing systems depending on the endpoint.
# Platform routing (na1, euw1) is used for ladders and summoner profiles.
# Regional routing (americas, europe) is used for match histories.
PLATFORM = "euW1" 
REGION = "europe"

## Ratelimit helper

In [3]:
def make_api_request(url):

    #A wrapper for requests.get() that automatically handles standard Riot API rate limits.
    while True:
        response = requests.get(url, headers=headers)
        
        if response.status_code == 200:
            return response.json()
            
        elif response.status_code == 429: # Too Many Requests 
            retry_after = int(response.headers.get("Retry-After", 10)) 
            print(f"Rate limit hit. Sleeping for {retry_after} seconds...")
            time.sleep(retry_after)
            
        elif response.status_code in [403, 401]:
            print("API Key expired or invalid. Please check your Riot Developer portal.")
            return None
            
        else:
            print(f"Error {response.status_code} for URL: {url}")
            return None

## Challenger ladder

In [ ]:
def get_challenger_puuids(limit=10):
    #Pulls the top Ranked Solo Challenger players and retrieves their PUUIDs.

    print("Fetching Challenger Ladder...")
    ladder_url = f"https://{PLATFORM}.api.riotgames.com/lol/league/v4/challengerleagues/by-queue/RANKED_SOLO_5x5"
    ladder_data = make_api_request(ladder_url)
    
    if not ladder_data:
        return []

    # Get the top X players from the ladder entries
    entries = ladder_data.get('entries', [])[:limit]
    
    puuids = []
    print(f"Extracting PUUIDs for the top {limit} players...")
    
    for entry in entries:
        # Riot's newer API data structures use PUUID
        if 'puuid' in entry:
            puuids.append(entry['puuid'])
            
        # Fallback just in case some legacy data still relies on summonerId
        elif 'summonerId' in entry:
            summoner_id = entry['summonerId']
            sum_url = f"https://{PLATFORM}.api.riotgames.com/lol/summoner/v4/summoners/{summoner_id}"
            sum_data = make_api_request(sum_url)
            
            if sum_data and 'puuid' in sum_data:
                puuids.append(sum_data['puuid'])
                
            time.sleep(1.2) 
        
    return puuids

# Test the ladder approach
ladder_seed_puuids = get_challenger_puuids(limit=50)
print(f"Successfully grabbed {len(ladder_seed_puuids)} PUUIDs from the ladder.")

print("Discovered PUUIDs:")
for p in list(ladder_seed_puuids):
    print(p) 

## Snowball Method

In [ ]:
def snowball_players_from_seed(seed_puuid, match_count=5):

    #Takes a single player, gets their recent matches, and extracts all other players from those matches.
    # queue=420 restricts the search to Ranked Solo/Duo games
    
    matches_url = f"https://{REGION}.api.riotgames.com/lol/match/v5/matches/by-puuid/{seed_puuid}/ids?queue=420&start=0&count={match_count}"
    match_ids = make_api_request(matches_url)
    
    if not match_ids:
        print("No matches found or error occurred.")
        return set()

    discovered_puuids = set() # Using a set prevents duplicate players
    
    print(f"Found {len(match_ids)} matches. Scraping other players...")
    
    for match_id in match_ids:
        match_url = f"https://{REGION}.api.riotgames.com/lol/match/v5/matches/{match_id}"
        match_data = make_api_request(match_url)
        
        if match_data and 'metadata' in match_data:
            # The metadata block contains a clean list of all 10 participants' PUUIDs
            participants = match_data['metadata']['participants']
            for p in participants:
                discovered_puuids.add(p)
                
        time.sleep(1.2) # Rate limit safety

    # Remove the seed player from the list of newly discovered ones (optional)
    if seed_puuid in discovered_puuids:
        discovered_puuids.remove(seed_puuid)
        
    return discovered_puuids

# Test the snowball approach using the first PUUID we got from the ladder above
if ladder_seed_puuids:
    seed = ladder_seed_puuids[0]
    print(f"Snowballing from seed PUUID: {seed[:15]}...")
    new_players = snowball_players_from_seed(seed, match_count=3)
    print(f"Discovered {len(new_players)} new players to analyze!")

    print("Discovered PUUIDs:")
    for p in list(new_players):
        print(p) 